# P3 分類と抽出の指標

> **この教材に登場する会社・人物・製品・数値・文書はすべて架空のものです。実在の企業・団体・個人とは関係ありません。**

ソラハ電機の問い合わせ分類（6 ラベル）と、ツバクロ少額短期保険の請求書の項目抽出（6 項目）を、混同行列・適合率と再現率・マクロ F1・自信の較正・厳密な照合とそろえた照合で測ります。応答は講師が Opus 5.5 / Sonnet 5.5（effort high）で回した本物の記録です。

- 上から順に実行します。0 のセルは準備です（Colab でも手元の Jupyter でも同じ）。
- API キーが無ければ講師の記録の再生（replay）で、本物の実行と同じ数字が出ます（費用 0）。
- 🔴 データはすべて架空の会社・文書です。

## 0. 準備

In [ ]:
# 0-1. 実行場所を見分けて、キットのフォルダに移る（Colab のときだけ取得とインストールをする）
import os, sys, subprocess
REPO_URL = "https://github.com/gucchi123/llm-eval-colab.git"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    KIT = "/content/llm-eval-kit"
    if not os.path.exists(KIT):
        subprocess.run(["git", "clone", "-q", REPO_URL, KIT], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic"], check=True)
else:  # 手元: notebooks/ から開いた前提。キットの根（common/ がある所）を探す
    KIT = os.path.abspath(".." if os.path.exists("../common/llm.py") else ".")
os.chdir(KIT)
sys.path.insert(0, KIT)
print("実行場所:", "Colab" if IN_COLAB else "手元の Jupyter", "/ キットの根:", os.path.basename(KIT))

In [ ]:
# 0-2. AI の呼び先を決める（キーは表示しない）
from common import nb
from common.nb import choose_provider, safe, show, show_file, lab
PROVIDER = choose_provider(IN_COLAB)

## 1. 分類の依頼文と、手引きの規則
依頼文は `prompts/classify.md`、手引きは `data/guide_inquiries.md`（長い行は折り返して表示）。

In [ ]:
from common import kit, llm as L, metrics as M, stats
for path, a, b in (("lab03_classify_extract/prompts/classify.md", 0, 7), ("data/guide_inquiries.md", 8, 12)):
    print("#", path)
    for line in open(path, encoding="utf-8").read().splitlines()[a:b]:
        while len(line) > 46:
            print("  ", line[:46]); line = "    " + line[46:]
        print("  ", line)

## 2. 一件を分類させる
到着から 60 日目の不具合の申し出。手引きの R2 では「修理・故障」です。

In [ ]:
rc = lab("run_classify", "lab03_classify_extract")
golden = kit.load_jsonl("golden_inquiries.jsonl")
hard = kit.load_jsonl("golden_inquiries_v2_hard.jsonl")
P = "replay" if PROVIDER == "mock" else PROVIDER
r = next(x for x in golden if x["id"] == "INQ-0004")
print(rc._render(r))
a = kit.parser("demo").parse_args(["--provider", P, "--model", "claude-opus-5-5"])
with safe():
    x = rc.classify_one(r, a)
    print("→ 正解", x["label"], "/ 予測", x["pred"], "/ 決め手", x["rule"], "/ 自信", x["confidence"])

## 3. 120 件を分類し、混同行列を作る（Opus 5.5）
行が正解、列が予測。対角線の外に数字があれば、それが取り違えです。

In [ ]:
RUNS = {}
def run_set(model, which):
    a = kit.parser("demo").parse_args(["--provider", P, "--model", model])
    return [rc.classify_one(r, a) for r in (golden if which == "v1" else hard)]
with safe():
    for m in ("claude-opus-5-5", "claude-sonnet-5-5"):
        for s in ("v1", "v2"):
            RUNS[m, s] = run_set(m, s)
    rep = rc.report(RUNS["claude-opus-5-5", "v1"])
    rows = [[t] + [rep["confusion"][t][p] or "." for p in rc.LABELS] for t in rc.LABELS]
    print(kit.table(rows, ["正解＼予測"] + [c[:2] for c in rc.LABELS]))
    k, n = sum(x["correct"] for x in RUNS["claude-opus-5-5", "v1"]), len(golden)
    print(f"正答 {k}/{n}  Wilson の 95% 区間 {kit.ci_str(*stats.wilson(k, n))}")

## 4. 適合率・再現率・F1 と、マクロ・マイクロの平均
`common/metrics.py` の本物のコードです。

In [ ]:
show(M.classification_report)

## 5. 難問 48 件（v2）のラベル別の表 ― 全問正解なのにマクロ F1 は 0.667
件数 0 のラベルは F1 が 0 になり、マクロ平均を引き下げます。

In [ ]:
rep2 = rc.report(RUNS["claude-opus-5-5", "v2"])
rows = [[c, v["support"], f"{v['precision']:.2f}", f"{v['recall']:.2f}", f"{v['f1']:.2f}"]
        for c, v in rep2["per_class"].items()]
print(kit.table(rows, ["ラベル", "件数", "適合率", "再現率", "F1"]))
print(f"正答率 {kit.pct(rep2['accuracy'])} / マイクロ F1 {rep2['micro_f1']:.3f} / マクロ F1 {rep2['macro_f1']:.3f}")
present = [v["f1"] for v in rep2["per_class"].values() if v["support"]]
print(f"件数のあるラベル {len(present)} 個だけで平均すると {sum(present) / len(present):.3f}")

## 6. 自信の較正（ECE）のコード
自信を 5 つの区間に分け、区間ごとの「平均の自信」と「実際の正答率」の差を、件数の重みで足します。

In [ ]:
show(M.ece)

## 7. 自信と正答のずれを測る ― 全問正解なのに自信は低め
実際の正答率は 1.00。自信の方が低いので、ずれは「自信が低すぎる」側です。

In [ ]:
for (m, s) in (("claude-opus-5-5", "v1"), ("claude-sonnet-5-5", "v1"), ("claude-opus-5-5", "v2")):
    rp = rc.report(RUNS[m, s])
    bins = [f"[{b['bin']}] {b['n']}件 自信{b['conf']:.2f} 正答{b['acc']:.2f}" for b in rp["reliability"] if b["n"]]
    print(f"{m.replace('claude-', ''):<11} {s}  ECE = {rp['ece']:.3f}")
    for b in bins:
        print("     ", b)

## 8. 抽出 ― 一枚の請求書と、取り出した 6 項目
和暦（R8.9.8）・全角の数字・日額と請求額が並ぶ、紛らわしい一枚です。

In [ ]:
rx = lab("run_extract", "lab03_classify_extract")
claims = kit.load_jsonl("claims.jsonl")
d = next(x for x in claims if x["id"] == "CLM-006")
for line in d["text"].splitlines():
    while len(line) > 46:
        print(line[:46]); line = "  " + line[46:]
    print(line)
a = kit.parser("demo").parse_args(["--provider", P, "--model", "claude-opus-5-5"])
with safe():
    e = rx.extract_one(d, a)
    rows = [[f, repr(d["gold"][f]), repr(e["pred"].get(f))] for f in rx.FIELDS]
    print(kit.table(rows, ["項目", "正解", "Opus の出力"]))

## 9. そろえてから比べる ― normalized の照合のコード
全角→半角、「30,000 円」→ 30000、「2026/9/8」→「2026-09-08」。

In [ ]:
show(rx.norm, 12, 19)
print()
for f, v in (("claim_amount", "３０,０００円"), ("claim_amount", 30000), ("incident_date", "2026/9/8"),
             ("claimant", "篠宮 彩花 "), ("receipt_attached", "あり")):
    print(f"norm({f!r}, {v!r}) → {rx.norm(f, v)!r}")

## 10. 48 枚を 2 つのモデルで ― strict と normalized
比べ方で点が変わるなら、採点の仕方を疑います。

In [ ]:
EX = {}
with safe():
    for m in ("claude-sonnet-5-5", "claude-opus-5-5"):
        a = kit.parser("demo").parse_args(["--provider", P, "--model", m])
        EX[m] = [rx.extract_one(d, a) for d in claims]
    sc = {(m, k): M.field_scores([r["gold"] for r in EX[m]], [r["pred"] for r in EX[m]], rx.FIELDS,
                                 norm=rx.norm if k == "norm" else None) for m in EX for k in ("strict", "norm")}
    rows = [[f] + [kit.pct(sc[m, k]["per_field"][f]) for m in EX for k in ("strict", "norm")] for f in rx.FIELDS]
    print(kit.table(rows, ["項目", "Sonnet 厳密", "そろえて", "Opus 厳密", "そろえて"]))
    for m in EX:
        print(f"{m}: 全項目が合った請求書 {kit.pct(sc[m, 'strict']['doc_all_correct'])}（n={len(EX[m])}）")

## 11. 飽和 ― 満点の試験は何も言わない
3 つの試験で 2 つのモデルを並べます。差が残るのは費用だけです。

In [ ]:
rows = []
for name, get in (("分類 v1 120件", lambda m: RUNS[m, "v1"]), ("分類 v2 難問48件", lambda m: RUNS[m, "v2"]),
                  ("抽出 48件", lambda m: EX[m])):
    for m in ("claude-sonnet-5-5", "claude-opus-5-5"):
        rr = get(m)
        k = sum(x["correct"] for x in rr) if "correct" in rr[0] else sum(x["pred"] == x["gold"] for x in rr)
        lo, _ = stats.wilson(k, len(rr))
        rows.append([name, m.replace("claude-", ""), f"{k}/{len(rr)}", f"{100 * lo:.1f}%", f"{sum(x['cost_usd'] for x in rr):.4f}"])
print(kit.table(rows, ["試験", "モデル", "正答", "区間の下限", "費用 USD"]))

## 12. 例題 ― 小さな混同行列から F1 を手で出す
10 件の小さな例。返品・交換 5 件、修理・故障 5 件。

In [ ]:
truth = ["返品・交換"] * 5 + ["修理・故障"] * 5
pred = ["返品・交換"] * 4 + ["修理・故障"] + ["返品・交換"] * 2 + ["修理・故障"] * 3
for c in ("返品・交換", "修理・故障"):
    tp = sum(t == c and p == c for t, p in zip(truth, pred))
    fp = sum(t != c and p == c for t, p in zip(truth, pred))
    fn = sum(t == c and p != c for t, p in zip(truth, pred))
    pr, re_ = tp / (tp + fp), tp / (tp + fn)
    print(f"{c}: TP={tp} FP={fp} FN={fn}  適合率 {tp}/{tp + fp}={pr:.3f}  再現率 {tp}/{tp + fn}={re_:.3f}"
          f"  F1={2 * pr * re_ / (pr + re_):.3f}")
rep = M.classification_report(truth, pred, ["返品・交換", "修理・故障"])
print("metrics の答え:", {c: round(v["f1"], 3) for c, v in rep["per_class"].items()},
      f"マクロ F1 {rep['macro_f1']:.3f} / 正答率 {rep['accuracy']:.1f}")